# 03 Run 405841 Segments (Part 2 / RegGS)

Goal:

1. run RegGS on `405841` prepared segmented scenes;
2. select one scene manually (`A` / `B` / `C`) and run only that one;
3. keep the infer / refine / metric pipeline unchanged;
4. keep a clean notebook entry for reproducible single-scene runs.

This notebook consumes data prepared by `01c_prepare_reggs_scene_405841_3segments.ipynb`.

In [1]:
from pathlib import Path
import json
import subprocess
import shutil
import copy
import yaml
from PIL import Image

CV_ROOT = Path('~/CV_Project').expanduser()
REGGS_ROOT = CV_ROOT / 'third_party' / 'RegGS'
REGGS_PYTHON = Path('/home/bzhang512/miniconda3/envs/reggs/bin/python')
PART2_ROOT = CV_ROOT / 'part2'
CONFIG_ROOT = PART2_ROOT / 'configs'
OUTPUT_ROOT = CV_ROOT / 'output' / 'part2'
RESULTS_ROOT = CV_ROOT / 'results' / 'part2'

SCENE_GROUP_ROOT = CV_ROOT / 'dataset' / '405841' / 'part2' / 'reggs_405841_3seg_25_70'
SCENE_OPTIONS = {
    'A': 'scene_A',
    'B': 'scene_B',
    'C': 'scene_C',
}

for d in [CONFIG_ROOT, OUTPUT_ROOT, RESULTS_ROOT]:
    d.mkdir(parents=True, exist_ok=True)

print('REGGS_ROOT =', REGGS_ROOT)
print('REGGS_PYTHON =', REGGS_PYTHON)
print('SCENE_GROUP_ROOT =', SCENE_GROUP_ROOT)
print('SCENE_OPTIONS =', SCENE_OPTIONS)

REGGS_ROOT = /home/bzhang512/CV_Project/third_party/RegGS
REGGS_PYTHON = /home/bzhang512/miniconda3/envs/reggs/bin/python
SCENE_GROUP_ROOT = /home/bzhang512/CV_Project/dataset/405841/part2/reggs_405841_3seg_25_70
SCENE_OPTIONS = {'A': 'scene_A', 'B': 'scene_B', 'C': 'scene_C'}


## 1. Run configuration (manual scene selection)


In [ ]:
# Manual scene switch: choose one from {'A', 'B', 'C'} and run only that scene.
SCENE_KEY = 'A'

assert SCENE_KEY in SCENE_OPTIONS, f'Invalid SCENE_KEY={SCENE_KEY}; choose one of {list(SCENE_OPTIONS.keys())}'
SCENE_NAME = SCENE_OPTIONS[SCENE_KEY]
SCENE_ROOT = SCENE_GROUP_ROOT / SCENE_NAME
SCENE_PARENT = SCENE_ROOT.parent

# Keep the same pipeline knobs; only scene address/selection is changed.
FORMAL_SAMPLE_RATE = 10
FORMAL_N_VIEWS = 4
FORMAL_NEW_SUBMAP_EVERY = 2

# 405841 has no dedicated ckpt: manual override > existing candidate fallback
FORMAL_NOPO_CHECKPOINT = None  # e.g. 'dl3dv' / 're10k' / 'acid' / None
NOPO_CHECKPOINT_CANDIDATES = ['dl3dv', 're10k', 'acid']
VALID_NOPO_CHECKPOINTS = {'re10k', 'dl3dv', 'acid'}
NOPO_CHECKPOINT_TO_FILE = {
    're10k': 're10k.ckpt',
    'dl3dv': 'mixRe10kDl3dv.ckpt',
    'acid': 'acid.ckpt',
}

if not NOPO_CHECKPOINT_CANDIDATES:
    raise ValueError('NOPO_CHECKPOINT_CANDIDATES cannot be empty.')

for ckpt_name in NOPO_CHECKPOINT_CANDIDATES:
    if ckpt_name not in VALID_NOPO_CHECKPOINTS:
        raise ValueError(
            f'Invalid candidate ckpt={ckpt_name}; valid options are {sorted(VALID_NOPO_CHECKPOINTS)}'
        )

if FORMAL_NOPO_CHECKPOINT is not None:
    if FORMAL_NOPO_CHECKPOINT not in VALID_NOPO_CHECKPOINTS:
        raise ValueError(
            f'Invalid FORMAL_NOPO_CHECKPOINT={FORMAL_NOPO_CHECKPOINT}; '
            f'valid options are {sorted(VALID_NOPO_CHECKPOINTS)}'
        )
    RESOLVED_NOPO_CHECKPOINT = FORMAL_NOPO_CHECKPOINT
    nopo_source = 'manual'
else:
    existing_candidates = [
        ck for ck in NOPO_CHECKPOINT_CANDIDATES
        if (REGGS_ROOT / 'pretrained_weights' / NOPO_CHECKPOINT_TO_FILE[ck]).exists()
    ]
    if existing_candidates:
        RESOLVED_NOPO_CHECKPOINT = existing_candidates[0]
        nopo_source = 'existing_candidate_fallback'
    else:
        RESOLVED_NOPO_CHECKPOINT = NOPO_CHECKPOINT_CANDIDATES[0]
        nopo_source = 'candidate_fallback'
        print(
            f'[WARN] none of NOPO_CHECKPOINT_CANDIDATES exists under pretrained_weights; '
            f'fallback to {RESOLVED_NOPO_CHECKPOINT}.'
        )

NOPO_WEIGHT_PATH = REGGS_ROOT / 'pretrained_weights' / NOPO_CHECKPOINT_TO_FILE[RESOLVED_NOPO_CHECKPOINT]
if not NOPO_WEIGHT_PATH.exists():
    print(f'[WARN] selected ckpt file not found: {NOPO_WEIGHT_PATH}')
else:
    print(f'[OK] selected ckpt file found: {NOPO_WEIGHT_PATH}')

RUN_TAG = f'reggs_405841_{SCENE_NAME}_{RESOLVED_NOPO_CHECKPOINT}-ckpt_sr{FORMAL_SAMPLE_RATE}_nv{FORMAL_N_VIEWS}'
RUN_OUTPUT = OUTPUT_ROOT / '405841' / RUN_TAG
CONFIG_PATH = CONFIG_ROOT / f'{RUN_TAG}.yaml'

print('SCENE_KEY =', SCENE_KEY)
print('SCENE_NAME =', SCENE_NAME)
print('SCENE_ROOT =', SCENE_ROOT)
print('NOPO_CHECKPOINT_CANDIDATES =', NOPO_CHECKPOINT_CANDIDATES)
print('RESOLVED_NOPO_CHECKPOINT =', RESOLVED_NOPO_CHECKPOINT)
print('nopo selection source =', nopo_source)
print('RUN_TAG =', RUN_TAG)
print('RUN_OUTPUT =', RUN_OUTPUT)
print('CONFIG_PATH =', CONFIG_PATH)

SCENE_KEY = A
SCENE_NAME = scene_A
SCENE_ROOT = /home/bzhang512/CV_Project/dataset/405841/part2/reggs_405841_3seg_25_70/scene_A
RUN_TAG = reggs_405841_scene_A_sr10_nv4
RUN_OUTPUT = /home/bzhang512/CV_Project/output/part2/405841/reggs_405841_scene_A_sr10_nv4
CONFIG_PATH = /home/bzhang512/CV_Project/part2/configs/reggs_405841_scene_A_sr10_nv4.yaml


## 2. Basic scene checks


In [3]:
assert SCENE_GROUP_ROOT.exists(), f'Missing segmented scene root: {SCENE_GROUP_ROOT}'
assert SCENE_ROOT.exists(), f'Missing selected scene root: {SCENE_ROOT}'
assert (SCENE_ROOT / 'images').exists(), 'Missing images dir'
assert (SCENE_ROOT / 'intrinsics.json').exists(), 'Missing intrinsics.json'
assert (SCENE_ROOT / 'cameras.json').exists(), 'Missing cameras.json'

cameras = json.loads((SCENE_ROOT / 'cameras.json').read_text(encoding='utf-8'))
intrinsics = json.loads((SCENE_ROOT / 'intrinsics.json').read_text(encoding='utf-8'))
print('selected scene =', SCENE_NAME)
print('num frames =', len(cameras))
print('intrinsics =', intrinsics)

selected scene = scene_A
num frames = 25
intrinsics = {'fx': 1.0764049814673433, 'fy': 1.6146074722010149, 'cx': 0.4950787903203501, 'cy': 0.5009273860525132}


## 3. Build formal config


In [ ]:
base_cfg_path = REGGS_ROOT / 'config' / 're10k.yaml'
base_cfg = yaml.safe_load(base_cfg_path.read_text(encoding='utf-8'))

cfg = copy.deepcopy(base_cfg)
cfg['dataset_name'] = 're10k'
cfg['n_views'] = int(FORMAL_N_VIEWS)
cfg['sample_rate'] = int(FORMAL_SAMPLE_RATE)
cfg['new_submap_every'] = int(FORMAL_NEW_SUBMAP_EVERY)
cfg['nopo_checkpoint'] = RESOLVED_NOPO_CHECKPOINT
cfg['frame_limit'] = -1

cfg['data']['input_path'] = str(SCENE_PARENT)
cfg['data']['scene_name'] = SCENE_NAME
cfg['data']['output_path'] = str(RUN_OUTPUT)

image_paths = sorted((SCENE_ROOT / 'images').glob('*.png'))
assert image_paths, f'No images found in {SCENE_ROOT / "images"}'
with Image.open(image_paths[0]) as im:
    img_w, img_h = im.size

cfg['cam']['H'] = int(img_h)
cfg['cam']['W'] = int(img_w)
cfg['cam']['fx'] = float(intrinsics['fx'])
cfg['cam']['fy'] = float(intrinsics['fy'])
cfg['cam']['cx'] = float(intrinsics['cx'])
cfg['cam']['cy'] = float(intrinsics['cy'])
cfg['cam']['depth_scale'] = 5000.0

CONFIG_PATH.write_text(yaml.safe_dump(cfg, sort_keys=False), encoding='utf-8')
print('detected image size =', (img_w, img_h))
print('using intrinsics =', intrinsics)
print('using checkpoint =', RESOLVED_NOPO_CHECKPOINT)
print(CONFIG_PATH.read_text(encoding='utf-8'))


detected image size = (960, 640)
using intrinsics = {'fx': 1.0764049814673433, 'fy': 1.6146074722010149, 'cx': 0.4950787903203501, 'cy': 0.5009273860525132}
dataset_name: re10k
checkpoint_path: null
frame_limit: -1
seed: 0
nopo_enable: true
n_views: 4
sample_rate: 10
nopo_checkpoint: re10k
new_submap_every: 2
aligner:
  iterations: 200
  cam_rot_lr: 0.0002
  cam_trans_lr: 0.002
  filter_alpha: false
  alpha_thre: 0.98
  soft_alpha: true
  mask_invalid_depth: false
  key_frame_pose: nopo
data:
  input_path: /home/bzhang512/CV_Project/dataset/405841/part2/reggs_405841_3seg_25_70
  output_path: /home/bzhang512/CV_Project/output/part2/405841/reggs_405841_scene_A_sr10_nv4
  scene_name: scene_A
cam:
  H: 640
  W: 960
  fx: 1.0764049814673433
  fy: 1.6146074722010149
  cx: 0.4950787903203501
  cy: 0.5009273860525132
  crop_edge: 0
  depth_scale: 5000.0



## 4. Optional cleanup


In [5]:
RESET_RUN_OUTPUT = False
if RESET_RUN_OUTPUT and RUN_OUTPUT.exists():
    shutil.rmtree(RUN_OUTPUT)
    print('removed old run output:', RUN_OUTPUT)
else:
    print('RESET_RUN_OUTPUT =', RESET_RUN_OUTPUT)


RESET_RUN_OUTPUT = False


## 5. Formal split preview


In [6]:
import numpy as np
frame_ids = np.arange(len(cameras))
test_ids = frame_ids[int(FORMAL_SAMPLE_RATE / 2)::FORMAL_SAMPLE_RATE]
remain_ids = np.array([i for i in frame_ids if i not in test_ids])
train_ids = remain_ids[np.linspace(0, remain_ids.shape[0] - 1, FORMAL_N_VIEWS).astype(int)]

print('train_ids =', train_ids.tolist())
print('test_ids =', test_ids.tolist())
print('num_train =', len(train_ids))
print('num_test =', len(test_ids))


train_ids = [0, 8, 16, 24]
test_ids = [5, 15]
num_train = 4
num_test = 2


## 6. Run infer


In [7]:
RUN_INFER = True
infer_cmd = [str(REGGS_PYTHON), 'run_infer.py', str(CONFIG_PATH)]
print('infer_cmd =', ' '.join(infer_cmd))

if RUN_INFER:
    proc = subprocess.run(
        infer_cmd,
        cwd=str(REGGS_ROOT),
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    print(proc.stdout)
    print('infer returncode =', proc.returncode)
else:
    print('Set RUN_INFER=True to launch formal infer.')


infer_cmd = /home/bzhang512/miniconda3/envs/reggs/bin/python run_infer.py /home/bzhang512/CV_Project/part2/configs/reggs_405841_scene_A_sr10_nv4.yaml
Using re10k.ckpt
/home/bzhang512/CV_Project/third_party/RegGS/src/utils/nopo_utils.py:82: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full c

## 7. Run refine


In [8]:
RUN_REFINE = True
refine_cmd = [str(REGGS_PYTHON), 'run_refine.py', '--checkpoint_path', str(RUN_OUTPUT), '--config_path', str(CONFIG_PATH)]
print('refine_cmd =', ' '.join(refine_cmd))

if RUN_REFINE:
    proc = subprocess.run(
        refine_cmd,
        cwd=str(REGGS_ROOT),
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    print(proc.stdout)
    print('refine returncode =', proc.returncode)
else:
    print('Set RUN_REFINE=True after infer completes.')


refine_cmd = /home/bzhang512/miniconda3/envs/reggs/bin/python run_refine.py --checkpoint_path /home/bzhang512/CV_Project/output/part2/405841/reggs_405841_scene_A_sr10_nv4 --config_path /home/bzhang512/CV_Project/part2/configs/reggs_405841_scene_A_sr10_nv4.yaml
/home/bzhang512/CV_Project/third_party/RegGS/run_refine.py:61: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommen

## 8. Run metric


In [9]:
RUN_METRIC = True
metric_cmd = [str(REGGS_PYTHON), 'run_metric.py', '--checkpoint_path', str(RUN_OUTPUT), '--config_path', str(CONFIG_PATH)]
print('metric_cmd =', ' '.join(metric_cmd))

if RUN_METRIC:
    proc = subprocess.run(
        metric_cmd,
        cwd=str(REGGS_ROOT),
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    print(proc.stdout)
    print('metric returncode =', proc.returncode)
else:
    print('Set RUN_METRIC=True after refine completes.')


metric_cmd = /home/bzhang512/miniconda3/envs/reggs/bin/python run_metric.py --checkpoint_path /home/bzhang512/CV_Project/output/part2/405841/reggs_405841_scene_A_sr10_nv4 --config_path /home/bzhang512/CV_Project/part2/configs/reggs_405841_scene_A_sr10_nv4.yaml
[Evaluator] Using gaussian file: /home/bzhang512/CV_Project/output/part2/405841/reggs_405841_scene_A_sr10_nv4/gaussians/global_refined_gs.ply
Training Frames: [ 0  8 16 24]
Eval Frames: [ 5 15]
Evaluating train render...
Setting up [LPIPS] perceptual loss: trunk [vgg], v[0.1], spatial [off]
/home/bzhang512/miniconda3/envs/reggs/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/bzhang512/miniconda3/envs/reggs/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 a

## 9. Output check


In [10]:
print('run output exists =', RUN_OUTPUT.exists())
if RUN_OUTPUT.exists():
    for p in sorted(RUN_OUTPUT.iterdir())[:30]:
        print(' -', p.name)
else:
    print('No run output yet.')


run output exists = True
 - ate_aligned.json
 - config.yaml
 - estimated_c2w.ckpt
 - eval_test.json
 - eval_train.json
 - eval_traj.png
 - gaussians
 - gt
 - submaps
 - test
 - train
 - vis_align
 - vis_integrate
